# Message Placeholder with Conversation Memory

`MessagesPlaceholder` inserts previous chat messages into a prompt so the model can answer with context from the same session.

In [143]:
from dotenv import load_dotenv
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_openai import ChatOpenAI

## Setup

Load environment variables, create the chat model, and build a prompt that has a slot for prior messages.

In [144]:
# Load API keys from .env, such as OPENAI_API_KEY.
load_dotenv()

True

In [145]:
# Optional Groq models. Uncomment one of these if you want to use Groq instead of OpenAI.
# from langchain_groq import ChatGroq
# llm = ChatGroq(model='llama-3.1-8b-instant')
# llm = ChatGroq(model='llama-3.3-70b-versatile')

In [146]:
# This notebook uses OpenAI by default.
llm = ChatOpenAI(model='gpt-4o', temperature=0)

In [147]:
prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a friendly chatbot. Use the conversation history to continue naturally.'),
    # RunnableWithMessageHistory will fill this with messages from the current session.
    MessagesPlaceholder(variable_name='history'),
    ('human', '{input}'),
])

In [148]:
# The prompt now expects two variables: history and input.
prompt

ChatPromptTemplate(input_variables=['history', 'input'], input_types={'history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langchain_c

## Build the Chain

The chain formats the prompt, sends it to the model, and parses the response into plain text.

In [149]:
parser = StrOutputParser()
chain = prompt | llm | parser

In [150]:
# Dictionary where each session id gets its own independent chat history.
store = {}

## Add Message History

`get_history` returns the memory object for the requested session. If the session is new, it creates an empty history first.

In [151]:
def get_history(session_id: str):
    """Return the chat history for a session, creating it on first use."""
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [152]:
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_history,
    input_messages_key='input',
    history_messages_key='history',
)

In [153]:
# First message in this session: the model should store the name Arun in history.
response_1 = chain_with_history.invoke(
    {'input': 'Hello, I am Arun.'},
    config={'configurable': {'session_id': 'arun_123'}},
)
response_1

"Hi Arun! Nice to meet you. How's your day going so far?"

In [154]:
# Follow-up in the same session: the model can use the previous message.
response_2 = chain_with_history.invoke(
    {'input': 'What is my name?'},
    config={'configurable': {'session_id': 'arun_123'}},
)
response_2

'Your name is Arun. How can I assist you today?'

In [155]:
# Another follow-up in the same session, this time in Hindi.
response_3 = chain_with_history.invoke(
    {'input': 'Mera naam kya hai?'},
    config={'configurable': {'session_id': 'arun_123'}},
)
response_3

'Aapka naam Arun hai. Aap se baat karke achha laga! Aap kaise hain?'